# Assignment 2: Binary Classification using Deep Neural Network
## IMDB Movie Review Sentiment Analysis

**Objective:** Classify movie reviews into positive and negative sentiments using Deep Neural Networks and experiment with different hyperparameters.

**Dataset:** IMDB Movie Reviews Dataset

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Create output directory if it doesn't exist
import os
output_dir = 'outputs'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory created/verified: {os.path.abspath(output_dir)}")

## 2. Load and Explore IMDB Dataset

In [ ]:
# Load IMDB dataset from CSV
# IMPORTANT: Update the path to your CSV file location

# Try different encodings to handle special characters
import chardet

# Detect file encoding
with open('IMDB Dataset.csv', 'rb') as f:
    result = chardet.detect(f.read(100000))  # Read first 100KB
    detected_encoding = result['encoding']
    print(f"Detected encoding: {detected_encoding}")

# Try loading with detected encoding, fallback to latin-1 if fails
try:
    data = pd.read_csv('IMDB Dataset.csv', encoding=detected_encoding)
    print(f"✓ Successfully loaded with {detected_encoding} encoding")
except:
    print(f"Failed with {detected_encoding}, trying latin-1...")
    data = pd.read_csv('IMDB Dataset.csv', encoding='latin-1')
    print("✓ Successfully loaded with latin-1 encoding")

print("\nDataset columns:", data.columns.tolist())
print("Dataset shape:", data.shape)
print("\nSentiment distribution:")
print(data['sentiment'].value_counts())
print("\nFirst few rows:")
display(data.head(3))

# Map sentiment to binary values
data['label'] = data['sentiment'].map({'positive': 1, 'negative': 0})

# Get reviews and labels
reviews = data['review'].values
labels = data['label'].values

print(f"\nTotal reviews: {len(reviews)}")
print(f"Positive reviews: {np.sum(labels == 1)} ({np.mean(labels)*100:.2f}%)")
print(f"Negative reviews: {np.sum(labels == 0)} ({(1-np.mean(labels))*100:.2f}%)")

## 3. Text Preprocessing and Tokenization

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Hyperparameters for text processing
max_features = 10000  # Maximum number of words to keep
maxlen = 256  # Maximum review length

# Split the data first
X_train_text, X_test_text, y_train, y_test = train_test_split(
    reviews, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training samples: {len(X_train_text)}")
print(f"Test samples: {len(X_test_text)}")

# Initialize and fit tokenizer on training data only
tokenizer = Tokenizer(num_words=max_features, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

# Convert texts to sequences
X_train = tokenizer.texts_to_sequences(X_train_text)
X_test = tokenizer.texts_to_sequences(X_test_text)

# Pad sequences to fixed length
X_train_padded = pad_sequences(X_train, maxlen=maxlen, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test, maxlen=maxlen, padding='post', truncating='post')

print(f"\nPadded shapes:")
print(f"X_train: {X_train_padded.shape}")
print(f"X_test: {X_test_padded.shape}")
print(f"Vocabulary size: {len(tokenizer.word_index)}")

# Sample encoded review
print(f"\nSample review (original):")
print(X_train_text[0][:200] + '...')
print(f"\nSample review (encoded):")
print(X_train_padded[0][:50])

## 3. Data Exploration and Visualization

In [ ]:
# Review length distribution
train_lengths = [len(review) for review in X_train]
test_lengths = [len(review) for review in X_test]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(train_lengths, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_xlabel('Review Length (words)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Review Lengths (Training Set)', fontsize=14, fontweight='bold')
axes[0].axvline(np.mean(train_lengths), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {np.mean(train_lengths):.0f}')
axes[0].axvline(np.median(train_lengths), color='green', linestyle='--', 
                linewidth=2, label=f'Median: {np.median(train_lengths):.0f}')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].boxplot([train_lengths, test_lengths], labels=['Train', 'Test'])
axes[1].set_ylabel('Review Length (words)', fontsize=12, fontweight='bold')
axes[1].set_title('Review Length Comparison', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/assignment2_review_lengths.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Review Length Statistics:")
print(f"Mean: {np.mean(train_lengths):.2f}")
print(f"Median: {np.median(train_lengths):.2f}")
print(f"Min: {np.min(train_lengths)}")
print(f"Max: {np.max(train_lengths)}")
print(f"Std: {np.std(train_lengths):.2f}")

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
train_counts = [np.sum(y_train == 0), np.sum(y_train == 1)]
axes[0].bar(['Negative', 'Positive'], train_counts, color=['#FF6B6B', '#4ECDC4'], 
            edgecolor='black', alpha=0.7)
axes[0].set_ylabel('Count', fontsize=12, fontweight='bold')
axes[0].set_title('Class Distribution - Training Set', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, count in enumerate(train_counts):
    axes[0].text(i, count, str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')

# Test set
test_counts = [np.sum(y_test == 0), np.sum(y_test == 1)]
axes[1].bar(['Negative', 'Positive'], test_counts, color=['#FF6B6B', '#4ECDC4'], 
            edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Count', fontsize=12, fontweight='bold')
axes[1].set_title('Class Distribution - Test Set', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, count in enumerate(test_counts):
    axes[1].text(i, count, str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/assignment2_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

## 5. Define Hyperparameter Configurations

In [ ]:
# Define simplified hyperparameter configurations
# We'll experiment with:
# 1. Different activation functions (ReLU, Tanh, Sigmoid)
# 2. Different batch sizes (32, 64, 128)

hyperparameter_configs = [
    {
        'name': 'Config 1: ReLU, Batch 32',
        'embedding_dim': 64,
        'dense_layers': [128, 64],
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 10,
        'activation': 'relu',
        'dropout': 0.3
    },
    {
        'name': 'Config 2: ReLU, Batch 64',
        'embedding_dim': 64,
        'dense_layers': [128, 64],
        'learning_rate': 0.001,
        'batch_size': 64,
        'epochs': 10,
        'activation': 'relu',
        'dropout': 0.3
    },
    {
        'name': 'Config 3: ReLU, Batch 128',
        'embedding_dim': 64,
        'dense_layers': [128, 64],
        'learning_rate': 0.001,
        'batch_size': 128,
        'epochs': 10,
        'activation': 'relu',
        'dropout': 0.3
    },
    {
        'name': 'Config 4: Tanh, Batch 32',
        'embedding_dim': 64,
        'dense_layers': [128, 64],
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 10,
        'activation': 'tanh',
        'dropout': 0.3
    },
    {
        'name': 'Config 5: Tanh, Batch 64',
        'embedding_dim': 64,
        'dense_layers': [128, 64],
        'learning_rate': 0.001,
        'batch_size': 64,
        'epochs': 10,
        'activation': 'tanh',
        'dropout': 0.3
    },
    {
        'name': 'Config 6: Sigmoid, Batch 32',
        'embedding_dim': 64,
        'dense_layers': [128, 64],
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 10,
        'activation': 'sigmoid',
        'dropout': 0.3
    },
]

print(f"Total configurations to test: {len(hyperparameter_configs)}")
print("\nWe are experimenting with:")
print("- Activation functions: ReLU, Tanh, Sigmoid")
print("- Batch sizes: 32, 64, 128")
print("- Fixed architecture: Embedding(64) -> Dense[128, 64]")
print("- Learning rate: 0.001 (constant)")
print("- Dropout: 0.3 (constant)")

## 6. Model Building Function

In [ ]:
def create_model(config):
    """
    Create a Deep Neural Network model for text classification.
    
    Args:
        config: Dictionary containing hyperparameters
    
    Returns:
        Compiled Keras model
    """
    model = keras.Sequential()
    
    # Embedding layer
    model.add(layers.Embedding(
        input_dim=max_features,
        output_dim=config['embedding_dim'],
        input_length=maxlen
    ))
    
    # Global Average Pooling to convert sequences to fixed-size vectors
    model.add(layers.GlobalAveragePooling1D())
    
    # Dense layers
    for units in config['dense_layers']:
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.Dropout(config['dropout']))
    
    # Output layer (sigmoid for binary classification)
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Compile model
    optimizer = keras.optimizers.Adam(learning_rate=config['learning_rate'])
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
    )
    
    return model

## 7. Train Models with Different Hyperparameters

In [ ]:
# Store results
results = []
histories = []
models = []

print("="*80)
print("TRAINING MODELS WITH DIFFERENT HYPERPARAMETER CONFIGURATIONS")
print("="*80)

for i, config in enumerate(hyperparameter_configs):
    print(f"\n{'='*80}")
    print(f"Training: {config['name']}")
    print(f"{'='*80}")
    print(f"Embedding Dim: {config['embedding_dim']}")
    print(f"Dense Layers: {config['dense_layers']}")
    print(f"Learning Rate: {config['learning_rate']}")
    print(f"Batch Size: {config['batch_size']}")
    print(f"Dropout: {config['dropout']}")
    
    # Create model
    model = create_model(config)
    
    # Print model summary for first configuration
    if i == 0:
        print("\nModel Architecture:")
        model.summary()
    
    # Callbacks
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=0
    )
    
    # Train model
    history = model.fit(
        X_train_padded, y_train,
        batch_size=config['batch_size'],
        epochs=config['epochs'],
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Evaluate on test set
    test_loss, test_acc, test_precision, test_recall = model.evaluate(
        X_test_padded, y_test, verbose=0
    )
    
    # Get predictions
    y_pred_prob = model.predict(X_test_padded, verbose=0).flatten()
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    # Calculate F1 score
    f1 = f1_score(y_test, y_pred)
    
    # Store results
    result = {
        'config_name': config['name'],
        'embedding_dim': config['embedding_dim'],
        'dense_layers': str(config['dense_layers']),
        'learning_rate': config['learning_rate'],
        'batch_size': config['batch_size'],
        'dropout': config['dropout'],
        'test_accuracy': test_acc,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'test_f1': f1,
        'test_loss': test_loss,
        'epochs_trained': len(history.history['loss'])
    }
    results.append(result)
    histories.append(history)
    models.append(model)
    
    print(f"\nResults:")
    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {test_precision:.4f}")
    print(f"  Recall:    {test_recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  Loss:      {test_loss:.4f}")
    print(f"  Epochs Trained: {len(history.history['loss'])}")

print(f"\n{'='*80}")
print("ALL MODELS TRAINED SUCCESSFULLY!")
print(f"{'='*80}")

## 8. Results Comparison

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('test_accuracy', ascending=False)
results_df.index = range(1, len(results_df) + 1)

print("\n" + "="*80)
print("PERFORMANCE COMPARISON OF ALL CONFIGURATIONS")
print("="*80)
print(results_df[['config_name', 'test_accuracy', 'test_precision', 'test_recall', 
                   'test_f1', 'epochs_trained']].to_string())

# Find best model
best_idx = results_df['test_accuracy'].idxmax() - 1
best_config = results[best_idx]

print(f"\n{'='*80}")
print(f"BEST PERFORMING CONFIGURATION")
print(f"{'='*80}")
print(f"Configuration: {best_config['config_name']}")
print(f"Embedding Dim: {best_config['embedding_dim']}")
print(f"Dense Layers: {best_config['dense_layers']}")
print(f"Learning Rate: {best_config['learning_rate']}")
print(f"Batch Size: {best_config['batch_size']}")
print(f"Dropout: {best_config['dropout']}")
print(f"\nPerformance Metrics:")
print(f"  Accuracy:  {best_config['test_accuracy']:.4f}")
print(f"  Precision: {best_config['test_precision']:.4f}")
print(f"  Recall:    {best_config['test_recall']:.4f}")
print(f"  F1 Score:  {best_config['test_f1']:.4f}")

## 9. Visualizations

In [ ]:
# Performance metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1']
titles = ['Accuracy Comparison', 'Precision Comparison', 'Recall Comparison', 'F1 Score Comparison']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for idx, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    ax = axes[idx // 2, idx % 2]
    
    sorted_df = results_df.sort_values(metric, ascending=True)
    
    bars = ax.barh(range(len(sorted_df)), sorted_df[metric], color=color, alpha=0.7, edgecolor='black')
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels([name.replace('Config ', 'C') for name in sorted_df['config_name']], fontsize=9)
    ax.set_xlabel(metric.replace('test_', '').title(), fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(0, 1)
    
    for i, (bar, value) in enumerate(zip(bars, sorted_df[metric])):
        ax.text(value, i, f' {value:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/assignment2_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Training history - Accuracy and Loss curves
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for idx, (history, config) in enumerate(zip(histories, hyperparameter_configs)):
    if idx < len(axes):
        ax = axes[idx]
        
        epochs = range(1, len(history.history['loss']) + 1)
        
        # Plot both accuracy and loss
        ax2 = ax.twinx()
        
        # Accuracy
        ln1 = ax.plot(epochs, history.history['accuracy'], 'b-', linewidth=2, 
                     label='Train Acc', alpha=0.8)
        ln2 = ax.plot(epochs, history.history['val_accuracy'], 'r-', linewidth=2, 
                     label='Val Acc', alpha=0.8)
        
        # Loss
        ln3 = ax2.plot(epochs, history.history['loss'], 'b--', linewidth=2, 
                      label='Train Loss', alpha=0.6)
        ln4 = ax2.plot(epochs, history.history['val_loss'], 'r--', linewidth=2, 
                      label='Val Loss', alpha=0.6)
        
        ax.set_xlabel('Epoch', fontsize=9)
        ax.set_ylabel('Accuracy', fontsize=9)
        ax2.set_ylabel('Loss', fontsize=9)
        ax.set_title(config['name'].replace('Config ', 'C'), fontsize=10, fontweight='bold')
        
        # Combined legend
        lns = ln1 + ln2 + ln3 + ln4
        labs = [l.get_label() for l in lns]
        ax.legend(lns, labs, loc='center right', fontsize=7)
        ax.grid(True, alpha=0.3)

# Remove extra subplots
for idx in range(len(histories), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('outputs/assignment2_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix for Best Model
best_model = models[best_idx]
y_pred_prob_best = best_model.predict(X_test_padded, verbose=0).flatten()
y_pred_best = (y_pred_prob_best > 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=axes[0],
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
axes[0].set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12, fontweight='bold')
axes[0].set_title(f'Confusion Matrix - {best_config["config_name"]}', 
                  fontsize=14, fontweight='bold')

# Normalized Confusion Matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', cbar=True, ax=axes[1],
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
axes[1].set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label', fontsize=12, fontweight='bold')
axes[1].set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/assignment2_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curve and AUC for Best Model
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob_best)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title(f'ROC Curve - {best_config["config_name"]}', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/assignment2_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Classification Report
print("="*80)
print(f"CLASSIFICATION REPORT - {best_config['config_name']}")
print("="*80)
print(classification_report(y_test, y_pred_best, 
                          target_names=['Negative', 'Positive']))

## 10. Analysis and Justification

In [ ]:
print("="*80)
print("ANALYSIS OF HYPERPARAMETER EXPERIMENTS")
print("="*80)

print("\n1. BEST PERFORMING CONFIGURATION:")
print(f"   {best_config['config_name']}")
print(f"   Accuracy: {best_config['test_accuracy']:.4f}, F1 Score: {best_config['test_f1']:.4f}")

print("\n2. KEY OBSERVATIONS:")

# Embedding dimension analysis
print(f"\n   a) Embedding Dimension:")
for dim in [32, 64, 128]:
    dim_configs = results_df[results_df['embedding_dim'] == dim]
    if not dim_configs.empty:
        print(f"      - {dim}D avg accuracy: {dim_configs['test_accuracy'].mean():.4f}")

# Network architecture
print(f"\n   b) Network Architecture:")
deep = results_df[results_df['config_name'].str.contains('Deep')]
wide = results_df[results_df['config_name'].str.contains('Wide')]
if not deep.empty:
    print(f"      - Deep Network accuracy: {deep['test_accuracy'].values[0]:.4f}")
if not wide.empty:
    print(f"      - Wide Network accuracy: {wide['test_accuracy'].values[0]:.4f}")

# Dropout analysis
print(f"\n   c) Dropout Effect:")
for dropout in [0.1, 0.3, 0.5]:
    dropout_configs = results_df[results_df['dropout'] == dropout]
    if not dropout_configs.empty:
        print(f"      - Dropout {dropout}: avg accuracy {dropout_configs['test_accuracy'].mean():.4f}")

# Batch size
print(f"\n   d) Batch Size:")
for bs in [64, 128]:
    bs_configs = results_df[results_df['batch_size'] == bs]
    if not bs_configs.empty:
        print(f"      - Batch size {bs}: avg accuracy {bs_configs['test_accuracy'].mean():.4f}")

print("\n3. CONCLUSION:")
print(f"   The best configuration achieves {best_config['test_accuracy']*100:.2f}% accuracy")
print(f"   on sentiment classification with balanced precision ({best_config['test_precision']:.4f})")
print(f"   and recall ({best_config['test_recall']:.4f}).")
print(f"   ROC-AUC Score: {roc_auc:.4f}")

## 11. Save Results

In [ ]:
# Save results to CSV
results_df.to_csv('outputs/assignment2_results.csv', index=False)
print("Results saved to 'assignment2_results.csv'")

# Save best model
best_model.save('outputs/assignment2_best_model.keras')
print("Best model saved to 'assignment2_best_model.keras'")